In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
from openai import OpenAI
from google.colab import userdata

In [ ]:
# FIX (Bug 2): Use the Instruct model, not the base model.
# The base model ("Meta-Llama-3.1-8B") won't reliably follow instructions
# or produce structured output. The Instruct variant was fine-tuned for that.
#
# FIX (H10 - frontier-vs-legacy generation parity): upgraded the Meta-family
# generator from Llama 3.1 (mid-2024) to the current Llama 4 generation, so it is
# a comparable *generation* to the frontier GPT generator instead of confounding
# "model family" with "model generation".
#   Llama 4 is MoE-only - there is no small dense Llama-4 8B - so the smallest
#   current-gen Llama is Scout (17B active / 109B total, 16 experts).
#   WARNING - HARDWARE: the 4-bit Scout checkpoint is ~55 GB and will NOT fit a
#   free Colab T4 (16 GB). Run this on an A100 80GB / H100 (Colab Pro+ or equiv).
from unsloth import FastLanguageModel
import torch

# FIX (Bug C5): max_seq_length must hold BOTH the input prompt and the generated
# output. The old 2048 cap left only ~48 tokens for the article once
# max_new_tokens reserved output space, truncating the article to nothing.
# Llama 4 Scout has a very long native context, so we raise the cap to feed whole
# arXiv articles. KV-cache memory scales with the tokens actually used (~2-3k for
# one article + a 400-token summary), not this ceiling, so the high cap is cheap.
max_seq_length = 131072
dtype = None          # Auto-detect: float16 for T4/V100, bfloat16 for Ampere+
load_in_4bit = True   # 4-bit quantization to fit in GPU memory

model, tokenizer = FastLanguageModel.from_pretrained(
    # Current-generation Meta model (Llama 4 Scout, Apr 2025), 4-bit via Unsloth.
    model_name = "unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


In [ ]:
# FIX (Bug 2 continued): LoRA / PEFT setup REMOVED.
# get_peft_model() adds trainable adapter weights — that's for fine-tuning,
# not inference. We're using the Instruct model as-is, so we skip this entirely.

# Instead, put the model into fast inference mode:
FastLanguageModel.for_inference(model)

In [ ]:
df=pd.read_csv("results_fetch_econ_full.csv")
print(f"Total rows in CSV: {len(df)}")

In [ ]:
df.head()

In [ ]:
# --- pipeline bootstrap: single source of truth is pipeline.py ---
# Makes pipeline.py importable whether running from a local repo checkout or in
# Google Colab, then imports the shared article fetch/clean helpers. Edit the
# fetch/clean logic ONCE in pipeline.py - not here, and not per-notebook.
import os, sys

def _ensure_pipeline_importable():
    try:
        import pipeline  # noqa: F401
        return
    except ImportError:
        pass
    # Local checkout: walk up from the CWD looking for pipeline.py
    here = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(here, "pipeline.py")):
            sys.path.insert(0, here)
            return
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    # Colab / fresh runtime: fetch pipeline.py from the repo's main branch
    import urllib.request
    url = "https://raw.githubusercontent.com/Dorothy99-love/Style-transfer/main/pipeline.py"
    urllib.request.urlretrieve(url, "pipeline.py")
    sys.path.insert(0, os.getcwd())

_ensure_pipeline_importable()
from pipeline import fetch_html_body_content, get_article_snippet_without_abstract


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

In [ ]:
import traceback
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
# from pipeline import fetch_html_body_content

# Initialize columns for the results
df["llama_prompt"] = ""  # Ensure the target column exists

In [ ]:
# --- Main evaluation loop ---
from transfer_prompt import transfer_prompt
for index, row in df.iterrows():
    try:
        html_url = row.get("html_url", None)
        if not html_url:
            continue

        # Fetch the content (assuming fetch_html_body_content is defined elsewhere)
        article, status = fetch_html_body_content(html_url)
        if status != "html_success":
            print(f"Row {index}: HTML fetch failed ({status})")
            continue

        article_snippet = get_article_snippet_without_abstract(article)
        prompt=transfer_prompt+"\n"+"Paper Content\n"+article_snippet
        # Build chat message structure
        messages = [
            {"role": "user", "content": prompt}
        ]

        encoded = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,
            truncation=True,
            return_dict=True
        )

        # move to device safely
        encoded = {k: v.to(device) for k, v in encoded.items()}

        outputs = model.generate(
            input_ids=encoded["input_ids"],
            attention_mask=encoded.get("attention_mask"),
            max_new_tokens=400,
            temperature=0.0,
            do_sample=False
        )

        # ✅ 正确截取生成部分
        input_length = encoded["input_ids"].shape[-1]
        generated_tokens = outputs[:, input_length:]

        output_text = tokenizer.decode(
            generated_tokens[0],
            skip_special_tokens=True
        ).strip()

        # Update the specific row's 'llama_response' column with the transferred text
        # Using .at[index, col] ensures we only modify the current row
        df.at[index, "llama_response"] = output_text

        print(f"--- Row {index}: Successfully processed ---")

    except Exception as e:
        # Catch and report errors to prevent the loop from breaking
        print(f"Error processing Row {index}: {e}")
        traceback.print_exc()
        continue

# Save the final results to CSV
df.to_csv("results_llama_all.csv", index=False)
print(f"\nSaved {len(df)} rows to results_llama_all.csv")